In [ ]:
from pathlib import Path

def find_sessions_with_empty_meta(root_dir):
    """
    Searches recursively for 'Meta' folders. 
    Returns the parent folder of any 'Meta' directory that contains no files.
    """
    root_path = Path(root_dir)
    sessions_to_process = []

    # rglob('Meta') finds all folders named Meta at any depth
    for meta_path in root_path.rglob('Meta'):
        if meta_path.is_dir():
            # Check if the directory is empty
            # We use next() to see if there is at least one item inside
            is_empty = not any(meta_path.iterdir())
            
            if is_empty:
                # .parent gives you the folder containing 'Meta'
                sessions_to_process.append(meta_path.parent)
                
    return sessions_to_process

# --- Usage ---
root = r"H:\Data\raw"
folders = find_sessions_with_empty_meta(root)

print(f"Found {len(folders)} sessions where Meta is empty:")
for f in folders:
    print(f"- {f}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import medfilt
from scipy.interpolate import interp1d
import spikeinterface.full as si

# Global settings
global_job_kwargs = dict(n_jobs=-3, chunk_duration="10s", progress_bar=True)
si.set_global_job_kwargs(**global_job_kwargs)

# --- YOUR ORIGINAL FUNCTIONS (Restored Exactly) ---

def find_sessions_with_empty_meta(root_dir):
    """
    Searches recursively for 'Meta' folders. 
    Returns the parent folder of any 'Meta' directory that contains no files.
    """
    root_path = Path(root_dir)
    sessions_to_process = []

    # rglob('Meta') finds all folders named Meta at any depth
    for meta_path in root_path.rglob('Meta'):
        if meta_path.is_dir():
            # Check if the directory is empty
            # We use next() to see if there is at least one item inside
            is_empty = not any(meta_path.iterdir())
            
            if is_empty:
                # .parent gives you the folder containing 'Meta'
                sessions_to_process.append(meta_path.parent)
                
    return sessions_to_process


def ttl_from_analog(signal, fs, hysteresis=0.1, filt_kernel=5):
    signal_f = medfilt(signal, kernel_size=filt_kernel)
    v_low = np.percentile(signal_f, 5)
    v_high = np.percentile(signal_f, 95)
    v_lo = v_low + hysteresis * (v_high - v_low)
    v_hi = v_high - hysteresis * (v_high - v_low)

    digital = np.zeros_like(signal_f, dtype=np.int8)
    state = 0
    for i, v in enumerate(signal_f):
        if state == 0 and v > v_hi: state = 1
        elif state == 1 and v < v_lo: state = 0
        digital[i] = state
    d = np.diff(digital)
    rising_idx = np.where(d == 1)[0] + 1
    falling_idx = np.where(d == -1)[0] + 1
    return digital, rising_idx, falling_idx

def quadrature_speed_direction(sigA, sigB, fs, pulses_per_rev=360, hysteresis=0.1):
    sig_len = len(sigA)
    time_full = np.arange(sig_len) / fs
    digA, rising_A, _ = ttl_from_analog(sigA, fs, hysteresis)
    digB, _, _ = ttl_from_analog(sigB, fs, hysteresis)
    if len(rising_A) < 2:
        return time_full * 1000, np.zeros(sig_len), np.zeros(sig_len)
    t_edges = rising_A / fs
    direction_edges = np.where(digB[rising_A] == 0, 1, -1)
    deg_per_edge = 360.0 / pulses_per_rev
    dtheta = direction_edges * deg_per_edge
    dt = np.diff(t_edges)
    speed_edges = np.diff(np.cumsum(dtheta)) / dt
    dir_edges = direction_edges[1:]
    t_mid = (t_edges[1:] + t_edges[:-1]) / 2
    f_speed = interp1d(t_mid, speed_edges, kind='previous', bounds_error=False, fill_value=0)
    f_dir = interp1d(t_mid, dir_edges, kind='previous', bounds_error=False, fill_value=0)
    return time_full * 1000, f_speed(time_full), f_dir(time_full)

def extract_ttl_from_bit(digital_word, bit, sampling_rate, plot_path, savename, plot=True):
    ttl_signal = (digital_word >> bit) & 1
    time_axis = np.arange(len(ttl_signal)) / sampling_rate
    if plot:
        plt.figure(figsize=(15, 3))
        plt.plot(time_axis, ttl_signal)
        plt.title(f'Isolated Bit {bit} ({savename})')
        plt.savefig(plot_path / f"ttl_bit_{bit}_{savename}.png")
        plt.close()
    diff = np.diff(ttl_signal)
    rising_idx = np.where(diff > 0)[0]
    falling_idx = np.where(diff < 0)[0]
    edges = np.concatenate([rising_idx / sampling_rate, falling_idx / sampling_rate])
    types = (['rising'] * len(rising_idx)) + (['falling'] * len(falling_idx))
    return pd.DataFrame({'timestamps': edges, 'edge_type': types}).sort_values('timestamps').reset_index(drop=True)

def extract_and_save_ttl_events(data, bit_name_pairs, save_path, plot_path):
    # CRITICAL: return_scaled=False so bitwise >> works on integers
    digital_signals = data.get_traces(return_scaled=False)
    digital_word = digital_signals[:, 8]
    sampling_rate = data.get_sampling_frequency()
    for bit, savename in bit_name_pairs:
        ttl_df = extract_ttl_from_bit(digital_word, bit, sampling_rate, plot_path, savename)
        ttl_df.to_csv(save_path / f"{savename}.csv", index=False)

# --- MAIN PROCESSING FUNCTION ---

def process_folder(folder_path):
    print(f"\n--- Processing: {folder_path.name} ---")
    base_path = Path(folder_path)
    metapath, plot_path, analyzerpath = base_path/'Meta', base_path/'plots', base_path/'analyzer'
    spike_times_path = base_path / 'spike_times'
    
    for p in [metapath, plot_path, spike_times_path]: p.mkdir(parents=True, exist_ok=True)

    # Load Data
    event = si.read_spikeglx(base_path, stream_id='nidq', load_sync_channel=False)
    analyzer = si.load_sorting_analyzer(analyzerpath)
    sf = event.get_sampling_frequency()

    # 1. Bombcell & Labels
    analyzer.compute(["spike_locations", "template_metrics", "quality_metrics"],n_jobs = 20, chunk_duration="1s" )
    bc_results = si.bombcell_label_units(analyzer, thresholds=si.bombcell_get_default_thresholds())
    labels = bc_results["bombcell_label"].values
    analyzer.sorting.set_property("bombcell_label", labels)
    
    # Save Labels Plot
    w = si.plot_unit_labels(analyzer, labels, ylims=(-300, 100))
    w.figure.savefig(plot_path / "bombcell_labels.png")
    plt.close(w.figure)

    # 2. Export Spike Times
    spikes = pd.DataFrame(analyzer.sorting.to_spike_vector())
    label_map = pd.Series(labels, index=range(len(analyzer.unit_ids)))
    spikes['label'] = spikes['unit_index'].map(label_map)
    spikes['time_s'] = spikes['sample_index'] / analyzer.sampling_frequency
    spike_df = spikes.drop(columns=['segment_index'], errors='ignore')
    spike_df.to_csv(spike_times_path / 'spike_times.csv', index=False)
    structured_array = spike_df.to_records(index=False)
    np.save(spike_times_path / 'spikes_times.npy', structured_array)

    # 3. Speed Detection (Quadrature)
    sigA = event.get_traces(channel_ids=[event.get_channel_ids()[6]]).squeeze()
    sigB = event.get_traces(channel_ids=[event.get_channel_ids()[5]]).squeeze()
    ts_ms, speed, direction = quadrature_speed_direction(sigA, sigB, sf, pulses_per_rev=225)
    df_speed = pd.DataFrame({'sample': ts_ms, 'speed': speed, 'direction': direction})
    df_speed.to_csv(metapath / 'speed.csv', index=False)

    # Save Speed Plot
    plt.figure(figsize=(10, 4))
    plt.plot(ts_ms[::6000]/1000, speed[::6000], color='black', linewidth=0.5)
    plt.title(f"Speed - {base_path.name}")
    plt.savefig(plot_path / "speed_plot.png")
    plt.close()

    # 4. Camera & Audio/State TTLs
    cam_signal = event.get_traces(channel_ids=[event.get_channel_ids()[3]]).squeeze()
    _, _, falling_A = ttl_from_analog(cam_signal, sf)
    if len(falling_A) > 0:
        pd.DataFrame({'camttl': [falling_A[0] / sf]}).to_csv(metapath / 'camttl.csv', index=False)

    pairs = [(1, 'State_changes'), (3, 'Audio')]
    extract_and_save_ttl_events(event, pairs, metapath, plot_path)

# --- EXECUTION ---
root = r"H:\Data\raw"
# Use your recursive discovery function here
folders = find_sessions_with_empty_meta(root) 

for f in folders:
    try:
        process_folder(f)
    except Exception as e:
        print(f"Failed on {f}: {e}")

In [ ]:
folder_path = r"H:\Data\raw\8251_recall_g0"
folder = Path(folder)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import medfilt
from scipy.interpolate import interp1d
import spikeinterface.full as si

def get_folders_with_nonempty_plots_no_bpod(base_path):
    """
    Returns subfolders where:
    - raw/subfolder_x/plots exists AND is not empty
    - AND BPOD.csv does NOT exist in subfolder_x
    """
    base_path = Path(base_path)
    valid_folders = []

    for subfolder in base_path.iterdir():
        if not subfolder.is_dir():
            continue
        
        plots_path = subfolder / "plots"
        bpod_file = subfolder / "BPOD.csv"
        
        if (
            plots_path.exists()
            and plots_path.is_dir()
            and any(p.is_file() for p in plots_path.iterdir())  # non-empty plots
            and not bpod_file.exists()  # BPOD.csv must NOT exist
        ):
            valid_folders.append(subfolder)
    print(len(valid_folders))

    return valid_folders

def redo_spiketimes_save(folder_path):
    print(f"\n--- Processing: {folder_path.name} ---")
    #folder_path = r"H:\Data\raw\8186_naive"
    base_path = Path(folder_path)
    analyzerpath =  base_path/'analyzer'
    spike_times_path = base_path / 'spike_times'
    # Global settings
    global_job_kwargs = dict(n_jobs=-3, chunk_duration="10s", progress_bar=True)
    si.set_global_job_kwargs(**global_job_kwargs)
    
    analyzer = si.load_sorting_analyzer(analyzerpath)
    bc_results = si.bombcell_label_units(analyzer, thresholds=si.bombcell_get_default_thresholds())
    labels = bc_results["bombcell_label"].values
    analyzer.sorting.set_property("bombcell_label", labels)
    spikes = pd.DataFrame(analyzer.sorting.to_spike_vector())
    label_map = pd.Series(labels, index=range(len(analyzer.unit_ids)))
    spikes['label'] = spikes['unit_index'].map(label_map)
    spikes['time_s'] = spikes['sample_index'] / analyzer.sampling_frequency
    spike_df = spikes.drop(columns=['segment_index'], errors='ignore')
    spike_df.to_csv(spike_times_path / 'spike_times.csv', index=False)
    
    
    # Convert to records and save
    structured_array = spike_df.to_records(index=False)
    
    
    np.save(spike_times_path / 'spike_times.npy', structured_array)

root = r"H:\Data\raw"
# Use your recursive discovery function here
folders = get_folders_with_nonempty_plots_no_bpod(root)

for f in folders:
    try:
        redo_spiketimes_save(f)
    except Exception as e:
        print(f"Failed on {f}: {e}")

C:\Users\Freitag\si_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


7

--- Processing: 8251_recall_g0 ---

--- Processing: 8181_naivel_g0 ---

--- Processing: 8272_recall_g0 ---

--- Processing: 8273_naive_g0 ---

--- Processing: 8272_g1 ---

--- Processing: 8181_recall_g0 ---

--- Processing: 8182_recall_g0 ---


In [ ]:
print(spike_df.dtypes)